<div style="padding: 20px; background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">📄 Module 6.1: Contextual Compression</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Shrinking documents down to only the exact sentences that matter.</p>
</div>

---

## 1. The "Needle in a Haystack" Problem

When you retrieve a 500-word chunk from a Vector Database, the answer to the user's question might only be **one single sentence** within that chunk.

Passing the other 480 irrelevant words to the LLM:
1. Wastes your context window.
2. Costs more money (if using paid APIs).
3. Confuses the LLM, increasing hallucination rates.

## 2. Contextual Compression
Contextual Compression solves this by placing a "Compressor" between the Vector DB and the LLM. 
The Compressor takes the retrieved chunks, analyzes them against the user's query, and **strips out all irrelevant sentences** *before* the final generation step.

### Course alignment and free-first stack

- Covers: Contextual compression with LLM extraction and embedding-based filtering.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))

# Notice how much irrelevant noise is in these documents!
docs = [
    Document(page_content=(
        "Python was created by Guido van Rossum. "
        "The language emphasises readability and simplicity. "
        "Python supports multiple programming paradigms. "
        "By the way, the Eiffel Tower is 330 metres tall. "
        "Python is widely used in data science and machine learning."
    )),
    Document(page_content=(
        "Machine learning is a subset of artificial intelligence. "
        "It enables computers to learn from data without explicit programming. "
        "The Sahara Desert is the world's largest hot desert. "
        "Supervised learning uses labelled training data."
    )),
]

vs = Chroma.from_documents(docs, embeddings, collection_name="compress_demo")
base_retriever = vs.as_retriever(search_kwargs={"k": 2})
query = "Who created Python?"

print("Baseline Retriever output (NO COMPRESSION):")
for d in base_retriever.invoke(query):
    print(f"\nRAW DOC: {d.page_content}")

## 3. Implementing the Custom LLM Extractor
Instead of using black-box LangChain components (which frequently deprecate), we will build our own robust Compressor using Groq's blazing-fast Llama 3 model. It will read the chunks and literally delete the sentences about the Eiffel Tower and the Sahara Desert.

In [ ]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    compressor_prompt = PromptTemplate.from_template(
        """Extract ONLY the information relevant to the question from the context.
        If the context has no relevant information, output exactly: NO_RELEVANT_DATA.
        
        Question: {question}
        Context: {context}
        
        Extracted Context:"""
    )
    
    chain = compressor_prompt | llm
    
    def compress_documents(query, docs):
        compressed = []
        for d in docs:
            res = chain.invoke({"question": query, "context": d.page_content})
            if "NO_RELEVANT_DATA" not in res.content:
                compressed.append(Document(page_content=res.content, metadata=d.metadata))
        return compressed
        
    # 1. Fetch raw documents
    raw_docs = base_retriever.invoke(query)
    
    # 2. Compress them!
    compressed_docs = compress_documents(query, raw_docs)
    
    print(f"Query: '{query}'")
    print("\nCOMPRESSED output:")
    for d in compressed_docs:
        print(f"\nCLEAN DOC: {d.page_content}")
else:
    print("GROQ_API_KEY not found. Please add it to your .env file to run the compressor.")